# Sovereign Byte Firewall - Kaggle GPU Training
Make sure GPU accelerator is ON: Settings > Accelerator > GPU T4 x2 or P100

In [ ]:
import subprocess, sys
for pkg in ['scapy', 'huggingface_hub>=0.25']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=True)
print('Dependencies installed')

In [ ]:
import os, subprocess
REPO_DIR = '/kaggle/working/sovereign-byte-firewall'
if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/TheSPST/sovereign-byte-firewall.git', REPO_DIR], check=True)
os.chdir(REPO_DIR)
print('Repo ready at:', os.getcwd())

In [ ]:
import os
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
    login(token=hf_token, add_to_git_credential=False)
    print('Logged in via Kaggle Secret')
except Exception:
    HF_TOKEN = ''  # paste token here as fallback
    if HF_TOKEN:
        login(token=HF_TOKEN, add_to_git_credential=False)
        print('Logged in via token')
    else:
        print('WARNING: No HF token - checkpoints will NOT be uploaded')
os.environ['HF_REPO_ID'] = 'spst01/sovereign-byte-firewall'

In [ ]:
import os
from huggingface_hub import hf_hub_download

DATASET_PATH = '/kaggle/working/Wednesdaytruncated.gz'

# Check manual upload locations first
for p in ['/kaggle/input/sovereign-byte-dataset/Wednesdaytruncated.gz', '/kaggle/input/Wednesdaytruncated.gz']:
    if os.path.exists(p):
        DATASET_PATH = p
        print('Dataset found (manual upload):', DATASET_PATH)
        break
else:
    if not os.path.exists(DATASET_PATH):
        try:
            print('Downloading dataset from HF Hub...')
            DATASET_PATH = hf_hub_download(
                repo_id='spst01/sovereign-byte-dataset',
                filename='Wednesdaytruncated.gz',
                repo_type='dataset',
                local_dir='/kaggle/working/'
            )
            print('Dataset downloaded:', DATASET_PATH)
        except Exception as e:
            print('ERROR: Dataset not found:', e)
            print('Upload Wednesdaytruncated.gz via: Notebook Data tab -> Add data -> Upload')
            raise
    else:
        print('Dataset found:', DATASET_PATH)

print(f'Size: {os.path.getsize(DATASET_PATH)/1e9:.2f} GB')

In [ ]:
import torch, subprocess
if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected! Turn on GPU accelerator in Settings.')
for i in range(torch.cuda.device_count()):
    name = torch.cuda.get_device_name(i)
    vram = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f'GPU[{i}]: {name} - {vram:.1f}GB VRAM')
subprocess.run(['nvidia-smi'])

In [ ]:
import sys, os
sys.path.insert(0, '/kaggle/working/sovereign-byte-firewall')
os.chdir('/kaggle/working/sovereign-byte-firewall')

sys.argv = [
    'run_kaggle.py',
    '--dataset_path', DATASET_PATH,
    '--epochs', '5',
    '--batch_size', '32',
    '--max_sequence_length', '512',
    '--lr', '5e-4',
    '--use_focal_loss', 'true',
    '--focal_gamma', '2.0',
    '--checkpoints_dir', '/kaggle/working/checkpoints',
]

from run_kaggle import main
main()

In [ ]:
import glob
ckpts = glob.glob('/kaggle/working/checkpoints/*.pt')
print(f'{len(ckpts)} checkpoint(s) saved:')
for c in sorted(ckpts):
    print(f'  {os.path.basename(c)} ({os.path.getsize(c)/1e6:.1f} MB)')
print('View on HF Hub: https://huggingface.co/spst01/sovereign-byte-firewall')